## Gold — Demand Metrics
Computes daily and weekly units sold, revenue, and return rate per product.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

### Read Enriched Silver

In [0]:
df = spark.read.table("databricks_cat.silver.orders_demand_silver")
df.printSchema()

### Daily Demand Metrics per Product

In [0]:
df_daily = (
    df
    .groupBy("order_date", "product_id", "product_name", "category", "brand")
    .agg(
        sum("quantity").alias("units_sold"),
        sum("total_amount").alias("daily_revenue"),
        count("order_id").alias("order_count"),
        sum("is_return").alias("return_count"),
        (sum("is_return") / count("order_id") * 100).alias("return_rate_pct")
    )
)
df_daily.display()

### Upsert Daily Demand → Gold

In [0]:
gold_daily_path = "abfss://gold@databricksetestorage.dfs.core.windows.net/DailyDemand"

if spark.catalog.tableExists("databricks_cat.gold.DailyDemand"):
    dt = DeltaTable.forName(spark, "databricks_cat.gold.DailyDemand")
    (
        dt.alias("trg")
        .merge(
            df_daily.alias("src"),
            "trg.order_date = src.order_date AND trg.product_id = src.product_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("✅ DailyDemand upserted.")
else:
    df_daily.write.format("delta").option("path", gold_daily_path).saveAsTable("databricks_cat.gold.DailyDemand")
    print("✅ DailyDemand created.")

### Weekly Demand Metrics per Product

In [0]:
df_weekly = (
    df
    .groupBy("year", "week", "product_id", "product_name", "category", "brand")
    .agg(
        sum("quantity").alias("weekly_units_sold"),
        sum("total_amount").alias("weekly_revenue"),
        count("order_id").alias("weekly_order_count"),
        (sum("is_return") / count("order_id") * 100).alias("weekly_return_rate_pct")
    )
)
df_weekly.display()

### Upsert Weekly Demand → Gold

In [0]:
gold_weekly_path = "abfss://gold@databricksetestorage.dfs.core.windows.net/WeeklyDemand"

if spark.catalog.tableExists("databricks_cat.gold.WeeklyDemand"):
    dt = DeltaTable.forName(spark, "databricks_cat.gold.WeeklyDemand")
    (
        dt.alias("trg")
        .merge(
            df_weekly.alias("src"),
            "trg.year = src.year AND trg.week = src.week AND trg.product_id = src.product_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("✅ WeeklyDemand upserted.")
else:
    df_weekly.write.format("delta").option("path", gold_weekly_path).saveAsTable("databricks_cat.gold.WeeklyDemand")
    print("✅ WeeklyDemand created.")

In [0]:
%sql
SELECT category, SUM(units_sold) AS total_units, ROUND(SUM(daily_revenue), 2) AS total_revenue
FROM databricks_cat.gold.DailyDemand
GROUP BY category
ORDER BY total_revenue DESC